In [12]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris['data']
y = iris['target']

names = iris['target_names']
feature_names = iris['feature_names']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [4]:
iris['target_names']

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [5]:
iris['feature_names']

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [10]:
std_scaler = StandardScaler()
std_scaler.fit(X_train, y_train)
X_train_tensor = torch.from_numpy(std_scaler.transform(X_train)).float()
X_test_tensor = torch.from_numpy(std_scaler.transform(X_test)).float()
y_train_tensor = torch.from_numpy(y_train).long()
y_test_tensor = torch.from_numpy(y_test).long()

print(X_train_tensor.shape, X_test_tensor.shape, y_train_tensor.shape, y_test_tensor.shape)

torch.Size([120, 4]) torch.Size([30, 4]) torch.Size([120]) torch.Size([30])


In [19]:
nb_epochs = 1000
minibatch_size = 120

In [14]:
class FunModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear_layers = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.LeakyReLU(0.1),
            nn.Linear(100, 20),
            nn.LeakyReLU(0.1),
            nn.Linear(20, 5),
            nn.LeakyReLU(0.1),
            nn.Linear(5, output_dim),
            # nn.Softmax(dim=-1)
            nn.LogSoftmax(dim=-1) # 최종 결과는 (6, 3) 이 되므로, dim=-1로 label 확률값이 들어 있는 마지막 차원을 지정해줘야 함 
        )
    def forward(self, x):
        y = self.linear_layers(x)
        return y

In [15]:
y_train_tensor.size()

torch.Size([120])

In [18]:
input_dim = X_train_tensor.size(-1)
output_dim = 3
print(input_dim, output_dim)

model = FunModel(input_dim, output_dim)

# loss_func = nn.CrossEntropyLoss() # softmax는 CrossEntropyLoss() 로 진행해야 함
loss_func = nn.NLLLoss() # log softmax는 NLLLoss()로 진행해야 함
optimizer = torch.optim.Adam(params=model.parameters())



4 3


In [20]:
for index in range(nb_epochs):
    indices = torch.randperm(X_test_tensor.size(0))
    
    x_batch_list = torch.index_select(X_train_tensor, 0, index=indices)
    y_batch_list = torch.index_select(y_train_tensor, 0, index=indices)
    x_batch_list = x_batch_list.split(minibatch_size, 0)
    y_batch_list = y_batch_list.split(minibatch_size, 0)
    
    epoch_loss = list()
    for x_minibatch, y_minibatch in zip(x_batch_list, y_batch_list):
        y_minibatch_pred = model(x_minibatch)
        loss = loss_func(y_minibatch_pred, y_minibatch)
        epoch_loss.append(loss)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if(index % 100) == 0:
        print(index, sum(epoch_loss)/len(epoch_loss))

0 tensor(1.1128, grad_fn=<DivBackward0>)
100 tensor(0.2829, grad_fn=<DivBackward0>)
200 tensor(0.0157, grad_fn=<DivBackward0>)
300 tensor(0.0033, grad_fn=<DivBackward0>)
400 tensor(0.0014, grad_fn=<DivBackward0>)
500 tensor(0.0008, grad_fn=<DivBackward0>)
600 tensor(0.0005, grad_fn=<DivBackward0>)
700 tensor(0.0004, grad_fn=<DivBackward0>)
800 tensor(0.0003, grad_fn=<DivBackward0>)
900 tensor(0.0002, grad_fn=<DivBackward0>)


### 테스트셋 기반 Evaluation

In [22]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    print(y_test_pred)
    y_pred_list = torch.argmax(y_test_pred, dim=1)

tensor([[-2.4614e-04, -8.3096e+00, -3.9266e+01],
        [-1.0522e+01, -1.3353e-02, -4.3247e+00],
        [-7.4980e-05, -9.4991e+00, -4.5910e+01],
        [-2.9266e+01, -1.0726e+01, -2.1934e-05],
        [-2.9046e+01, -1.0865e+01, -1.9073e-05],
        [-2.5034e-05, -1.0597e+01, -4.7146e+01],
        [-4.7597e+00, -8.6048e-03, -2.2550e+01],
        [-9.8563e+00, -5.2570e-05, -1.5807e+01],
        [-9.7751e-06, -1.1532e+01, -5.1911e+01],
        [-1.6995e+01, -9.5939e-01, -4.8309e-01],
        [-8.7491e+00, -2.7736e-04, -9.0394e+00],
        [-4.4876e+01, -1.6917e+01,  0.0000e+00],
        [-1.4424e-05, -1.1146e+01, -5.2203e+01],
        [-8.7615e-05, -9.3426e+00, -4.4372e+01],
        [-2.5767e+01, -9.1843e+00, -1.0263e-04],
        [-2.2172e+01, -6.2974e+00, -1.8428e-03],
        [-2.6305e+01, -9.1203e+00, -1.0943e-04],
        [-5.5431e-05, -9.8006e+00, -4.4679e+01],
        [-2.3197e+01, -8.4923e+00, -2.0502e-04],
        [-1.0962e+01, -1.0070e-01, -2.3457e+00],
        [-1.2246e+01

### mini-batch size 기반 예측

In [24]:
y_pred_list = list()
x_test_batch_list = X_test_tensor.split(minibatch_size, 0)

model.eval()
with torch.no_grad():
    for x_minibatch in x_test_batch_list:
        y_test_pred = model(x_minibatch)
        print(y_test_pred.shape)
        y_test_pred = torch.argmax(y_test_pred, dim=1)
        print(y_test_pred.shape)
        y_pred_list.extend(y_test_pred.detach().tolist())
        
y_pred_list = torch.tensor(y_pred_list)

torch.Size([30, 3])
torch.Size([30])


### Multi-Label Classification 기본 메트릭

- None : 라벨 별로 각 계산값 그대로 출력함
- micro : 전체 라벨 값을 합하여 계산함
- macro : 라벨 별로 계산된 값에 대한 전체 평균을 출력함

In [34]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print(confusion_matrix(y_test_tensor, y_pred_list))
print("Precision: " + str(precision_score(y_test_tensor, y_pred_list, average=None)))
print("Precision: " + str(precision_score(y_test_tensor, y_pred_list, average='macro'))) # Macro metric : 라벨 별로 계산된 값에 대한 전체 평균을 출력함
print("Precision: " + str(precision_score(y_test_tensor, y_pred_list, average='micro'))) # Micro metric : 전체 라벨 값을 합하여 계산함


print("Recall: "+ str(recall_score(y_test_tensor, y_pred_list, average=None)))
print("Recall: "+ str(recall_score(y_test_tensor, y_pred_list, average='macro')))
print("Recall: "+ str(recall_score(y_test_tensor, y_pred_list, average='micro')))

print("F1 SCORE: " + str(f1_score(y_test_tensor, y_pred_list, average=None)))  # Precision과 Recall 이 적절할 경우 F1 SCORE이 나온다.
print("F1 SCORE: " + str(f1_score(y_test_tensor, y_pred_list, average='macro')))  # Precision과 Recall 이 적절할 경우 F1 SCORE이 나온다.
print("F1 SCORE: " + str(f1_score(y_test_tensor, y_pred_list, average='micro')))  # Precision과 Recall 이 적절할 경우 F1 SCORE이 나온다.

[[11  0  0]
 [ 0  8  1]
 [ 0  0 10]]
Precision: [1.         1.         0.90909091]
Precision: 0.9696969696969697
Precision: 0.9666666666666667
Recall: [1.         0.88888889 1.        ]
Recall: 0.9629629629629629
Recall: 0.9666666666666667
F1 SCORE: [1.         0.94117647 0.95238095]
F1 SCORE: 0.9645191409897292
F1 SCORE: 0.9666666666666667
